In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!unzip -q "/content/drive/MyDrive/KhPI_HW/linear_algebra_intro_practice-main.zip" -d /content/idpp
!find /content/idpp -name "regularization.py"
!find /content/idpp -name "pyproject.toml"

/content/idpp/linear_algebra_intro_practice-main/regularization.py
/content/idpp/linear_algebra_intro_practice-main/pyproject.toml


In [4]:
%cd /content/idpp/linear_algebra_intro_practice-main

/content/idpp/linear_algebra_intro_practice-main


In [5]:
!pip -q install uv
!uv sync
!uv run pytest tests/test_regularization.py -v

Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 113 packages in 959ms
Prepared 109 packages in 7.82s
Installed 109 packages in 633ms
 + anyio==4.15.1
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==26.1.0
 + arrow==1.4.0
 + asttokens==3.0.2
 + async-lru==2.4.0
 + attrs==26.1.0
 + babel==2.18.0
 + beautifulsoup4==4.15.0
 + bleach==6.4.0
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + cloudpickle==3.1.2
 + comm==0.2.3
 + contourpy==1.4.0
 + cycler==0.12.1
 + debugpy==1.8.22
 + defusedxml==0.7.1
 + executing==2.3.0
 + fastjsonschema==2.22.2
 + fonttools==4.67.0
 + fqdn==1.6.0
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.20
 + iniconfig==2.3.1
 + ipykernel==7.3.0
 + ipython==9.17.1
 + ipython-pygments-lexers==1.1.1
 + isoduration==20.11.0
 + jedi==0.20.1
 + jinja2==3.1.6
 + joblib==1.6.0
 + json5==0.17.3
 + jsonpointer==3.6
 + jsonschema==4.26.0
 + jsonschema-specifications==2025.9.1
 + jupyter-cl

In [7]:
# Перезавантажуємо модуль, щоб Colab точно взяв останню версію regularization.py
import importlib
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score
import regularization

importlib.reload(regularization)

print("Тест регресії (Diabetes Dataset)---")
X_train_reg, X_test_reg, y_train_reg, y_test_reg = regularization.get_regression_data()

# Навчаємо моделі
lin_reg = regularization.linear_regression(X_train_reg, y_train_reg)
ridge_reg = regularization.ridge_regression(X_train_reg, y_train_reg)
lasso_reg = regularization.lasso_regression(X_train_reg, y_train_reg)

# Оцінка на тестових даних
models_reg = [("Linear Regression", lin_reg), ("Ridge (L2)", ridge_reg), ("Lasso (L1)", lasso_reg)]

for name, model in models_reg:
    preds = model.predict(X_test_reg)
    mse = mean_squared_error(y_test_reg, preds)
    r2 = r2_score(y_test_reg, preds)

    # витягнемо коеф. та гіперпараметри альфа (якщо є)
    alpha_val = getattr(model, 'alpha', 'N/A (Unregularised)')
    num_zero_coefs = np.sum(np.abs(model.coef_) < 1e-5)

    print(f"[{name}]")
    print(f"-> Alpha: {alpha_val}")
    print(f"-> MSE: {mse:.4f} | R2: {r2:.4f}")
    print(f"-> Кількість обнулених коефіцієнтів: {num_zero_coefs} з {len(model.coef_)}")
    print("-" * 50)


print("\nТестуємо класифікацію (Breast Cancer Dataset)---")
X_train_clf, X_test_clf, y_train_clf, y_test_clf = regularization.get_classification_data()

# Навчаємо моделі
log_unreg = regularization.logistic_regression(X_train_clf, y_train_clf)
log_l2 = regularization.logistic_l2_regression(X_train_clf, y_train_clf)
log_l1 = regularization.logistic_l1_regression(X_train_clf, y_train_clf)

models_clf = [("Logistic (Unregularised)", log_unreg), ("Logistic L2", log_l2), ("Logistic L1", log_l1)]

for name, model in models_clf:
    preds = model.predict(X_test_clf)
    acc = accuracy_score(y_test_clf, preds)
    f1 = f1_score(y_test_clf, preds)

    c_val = getattr(model, 'C', 'N/A (inf)')
    num_zero_coefs = np.sum(np.abs(model.coef_) < 1e-5)

    print(f"[{name}]")
    print(f"-> C parameter: {c_val}")
    print(f"-> Accuracy: {acc:.4f} | F1-Score: {f1:.4f}")
    print(f"-> Кількість обнулених коефіцієнтів: {num_zero_coefs} з {model.coef_.shape[1]}")
    print("-" * 50)

Тест регресії (Diabetes Dataset)---
[Linear Regression]
-> Alpha: N/A (Unregularised)
-> MSE: 2900.1936 | R2: 0.4526
-> Кількість обнулених коефіцієнтів: 0 з 10
--------------------------------------------------
[Ridge (L2)]
-> Alpha: 39.79993311036789
-> MSE: 2858.3579 | R2: 0.4605
-> Кількість обнулених коефіцієнтів: 0 з 10
--------------------------------------------------
[Lasso (L1)]
-> Alpha: 1.6732240802675582
-> MSE: 2800.3644 | R2: 0.4714
-> Кількість обнулених коефіцієнтів: 3 з 10
--------------------------------------------------

Тестуємо класифікацію (Breast Cancer Dataset)---
[Logistic (Unregularised)]
-> C parameter: inf
-> Accuracy: 0.9561 | F1-Score: 0.9640
-> Кількість обнулених коефіцієнтів: 0 з 30
--------------------------------------------------
[Logistic L2]
-> C parameter: 1.6608827826277148
-> Accuracy: 0.9737 | F1-Score: 0.9790
-> Кількість обнулених коефіцієнтів: 0 з 30
--------------------------------------------------
[Logistic L1]
-> C parameter: 0.9249147

# CONCLUSIONS

Було досліджено застосування методів регуляризації (Ridge/L2 та Lasso/L1) для подолання некоректно поставлених задач за Адамаром (коли модель перенавчається через шум або сильну кореляцію ознак). Нижче наведено результати експериментів та аналіз (за README "What to hand in").

1. Поведінка коефіцієнтів: L1 vs L2

Результати демонструють різницю у математичній природі штрафів L1 та L2:

- **L2-регуляризація (Ridge/Logistic L2):**
  - Мета: рівномірно й плавно зменшує значення коефіцієнтів, роблячи модель стійкою до мультиколінеарності.
  - Результат: **Жоден** коефіцієнт не став рівним нулю (0 з 10 для регресії, 0 з 30 для класифікації). Всі ознаки залишаються в моделі, але їхній вплив вже зважений.
- **L1-регуляризація (Lasso/Logistic L1):**
  - Мета: завдяки особливостям геометрії обмежень виступає ефективним інструментом відбору ознак (feature selection).
  - Результат: активно занулює коефіцієнти незначущих ознак. У нашому експерименті Lasso обнулила 3 з 10 ознак у регресії, а L1-логістична регресія скоротила простір ознак рівно вдвічі, обнуливши 15 з 30 параметрів.

---
2. Обрані гіперпараметри (α та C)

За допомогою `GridSearchCV` для кожного набору даних були знайдені наступні оптимальні параметри:
- Для задачі регресії (Diabetes Dataset):
  - **Ridge (L2):** α ≈ 39.80 (помірна регуляризація для плавного зменшення ваг + через квадратич. природу штрафу L2 вимагає більших значень α, щоб суттєво зменшити вплив великих ваг без їх повного обнулення).
  - **Lasso (L1):** α ≈ 1.67 (сильніший штраф + штраф L1 діє лін. та активніше, тому менше значення (порів. з поперед. пунктом) α дозволило виділити найважливіші ознаки).
- Для задачі класифікації (Breast Cancer Dataset):
  - **Logistic L2:** C ≈ 1.66 (де менший C означає сильнішу регуляризацію, оскільки C = 1/α).
  - **Logistic L1:** C ≈ 0.92.

---

3. Чому модель із штрафуванням може перевершити модель без штрафування на даних, яких вона раніше не бачила?

Як можемо побачити з порівняння показників якості підтверджується перевагв регуляризації:
- **Регресія:** MSE зменшився з `2900.19` (нерегуляризована) до `2858.36` (Ridge) і найкращого значення `2800.36` (Lasso), при цьому коефіцієнт детермінації R^2 зріс з `0.4526` до `0.4714`.
- **Класифікація:** Accuracy зросла з `0.9561` до `0.9737` (L2), а F1-Score збільшився з `0.9640` до `0.9790`.

Причина:

Нерегуляризовані моделі (C = ∞) намагаються мінімізувати помилку виключно на тренувальному наборі. Через це на зашумлених або сильно корельованих даних вони підлаштовуються під випадкові викиди (тобто спостерігаємо перенавчання/overfitting), що призводить до величезної варіативності коефіцієнтів.

Введення штрафу (L1 або L2) додає невеликий систематич. зсув (bias), але значно знижує дисперсію моделі. Це робить її стійкою до мікро-збурень на тестових даних, які модель ніколи раніше не бачила, що й призводить до нижчої похибки та вищої точності у реальних умовах.

# Код для regularization

In [ ]:
"""Practice 5: regularization as a cure for an ill-posed estimation problem.

Ordinary least squares on correlated, noisy features is ill-posed in the sense of
Hadamard: the solution is not stable under small perturbations of the data. Ridge
and Lasso restore stability by adding a penalty term. Your job is to build the six
estimators below, then write up the comparison (see README, "What to hand in").
"""

import numpy as np
from sklearn.base import BaseEstimator
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer, load_diabetes

ALPHA_GRID = np.linspace(1e-3, 1e2, 300)
C_GRID = np.logspace(-3, 2, 60)


def preprocess(X: np.ndarray, y: np.ndarray) -> list[np.ndarray]:
    """Split into train/test, then standardise using training statistics only.

    The split comes first on purpose. Fitting the scaler on the full matrix would
    leak test-set means and variances into training - a small, silent error that
    inflates every score you report afterwards.

    Args:
        X (np.ndarray): feature matrix.
        y (np.ndarray): target vector.

    Returns:
        list[np.ndarray]: [X_train, X_test, y_train, y_test], features standardised.
    """
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler().fit(X_train)

    return [scaler.transform(X_train), scaler.transform(X_test), y_train, y_test]


def get_regression_data() -> list[np.ndarray]:
    """Load and preprocess the diabetes dataset for regression tasks.

    Returns:
        list[np.ndarray]: [X_train, X_test, y_train, y_test].
    """
    data = load_diabetes()
    return preprocess(data.data, data.target)


def get_classification_data() -> list[np.ndarray]:
    """Load and preprocess the breast cancer dataset for classification tasks.

    Returns:
        list[np.ndarray]: [X_train, X_test, y_train, y_test].
    """
    data = load_breast_cancer()
    return preprocess(data.data, data.target)


def linear_regression(X: np.ndarray, y: np.ndarray) -> BaseEstimator:
    """Fit an unregularised least-squares model. This is your baseline.

    Args:
        X (np.ndarray): feature matrix.
        y (np.ndarray): target vector.

    Returns:
        BaseEstimator: fitted LinearRegression.
    """
    #raise NotImplementedError
    # Ініціалізуємо лін. регресію та навчаємо її на вхідних даних без регуляризації
    model = LinearRegression()
    return model.fit(X, y)

def ridge_regression(X: np.ndarray, y: np.ndarray) -> BaseEstimator:
    """Fit a ridge (L2) model, choosing `alpha` from ALPHA_GRID with GridSearchCV.

    Args:
        X (np.ndarray): feature matrix.
        y (np.ndarray): target vector.

    Returns:
        BaseEstimator: the refitted best estimator (`.best_estimator_`), so that
            the returned object exposes `.alpha` and `.coef_`.
    """
    # raise NotImplementedError
    # Шукаємо оптимальн. alpha для стабіліз. моделі
    grid = GridSearchCV(Ridge(), param_grid={'alpha': ALPHA_GRID}, cv=5, n_jobs=-1)
    grid.fit(X, y)
    return grid.best_estimator_

def lasso_regression(X: np.ndarray, y: np.ndarray) -> BaseEstimator:
    """Fit a lasso (L1) model, choosing `alpha` from ALPHA_GRID with GridSearchCV.

    Args:
        X (np.ndarray): feature matrix.
        y (np.ndarray): target vector.

    Returns:
        BaseEstimator: the refitted best estimator, exposing `.alpha` and `.coef_`.
    """
    #raise NotImplementedError
    # Збільшуємо max_iter для Lasso, щоб уникнути помилок збіжності на сильно корельов. ознаках
    grid = GridSearchCV(Lasso(max_iter=10000), param_grid={'alpha': ALPHA_GRID}, cv=5, n_jobs=-1)
    grid.fit(X, y)
    return grid.best_estimator_

def logistic_regression(X: np.ndarray, y: np.ndarray) -> BaseEstimator:
    """Fit logistic regression with no regularisation at all.

    Note: `penalty=` was deprecated in scikit-learn 1.8 and is removed in 1.10.
    Switch the penalty off with `C=np.inf` instead.

    Args:
        X (np.ndarray): feature matrix.
        y (np.ndarray): binary target vector.

    Returns:
        BaseEstimator: fitted LogisticRegression.
    """
    #raise NotImplementedError
    # Через C=np.inf вимикаємо регуляризацію + оптиміз. стійким розв'язув. saga
    model = LogisticRegression(C=np.inf, solver='saga', max_iter=10000, random_state=42)
    return model.fit(X, y)

def logistic_l2_regression(X: np.ndarray, y: np.ndarray) -> BaseEstimator:
    """Fit L2-penalised logistic regression, tuning `C` over C_GRID with GridSearchCV.

    Note: L2 is `l1_ratio=0` in scikit-learn >= 1.8 (not `penalty="l2"`).

    Args:
        X (np.ndarray): feature matrix.
        y (np.ndarray): binary target vector.

    Returns:
        BaseEstimator: the refitted best estimator, exposing `.C` and `.coef_`.
    """
    #raise NotImplementedError
    # Робимо звичайну L2-регуляризацію, але через ElasticNet з l1_ratio=0 для сумісності
    base_model = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0, max_iter=10000, random_state=42)
    grid = GridSearchCV(base_model, param_grid={'C': C_GRID}, cv=5, n_jobs=-1)
    grid.fit(X, y)
    return grid.best_estimator_

def logistic_l1_regression(X: np.ndarray, y: np.ndarray) -> BaseEstimator:
    """Fit L1-penalised logistic regression, tuning `C` over C_GRID with GridSearchCV.

    Note: L1 is `l1_ratio=1` in scikit-learn >= 1.8, and it needs a solver that
    supports it - `liblinear` or `saga`. The default `lbfgs` raises ValueError.

    Args:
        X (np.ndarray): feature matrix.
        y (np.ndarray): binary target vector.

    Returns:
        BaseEstimator: the refitted best estimator, exposing `.C` and `.coef_`.
    """
    #raise NotImplementedError
    # Задаємо L1-регуляризацію через ElasticNet (l1_ratio=1) для ефектив. відбору ознак за допомогою saga
    base_model = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=1, max_iter=10000, random_state=42)
    grid = GridSearchCV(base_model, param_grid={'C': C_GRID}, cv=5, n_jobs=-1)
    grid.fit(X, y)
    return grid.best_estimator_